# **Import Library**

In [1]:
!pip install wandb timm -q

import random
import os
import copy
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import wandb

from dotenv import load_dotenv
load_dotenv()

# Ambil API key dari environment variable
wandb_api_key = os.getenv("WANDB_API_KEY")

# Login ke wandb
wandb.login(key=wandb_api_key)

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
import random

from torchvision import transforms, datasets
import timm   # PERBAIKAN: ganti torchvision.models.resnet50 -> timm (untuk ViT)

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\UNIDA\_netrc.
wandb: Currently logged in as: devianestnarendra to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# **Dataset Path**

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

# FIX (Claude): cudnn.benchmark auto-tune algoritma konvolusi terbaik untuk
# ukuran input yang konsisten (semua gambar di-resize ke 224x224) -> speedup
# tambahan di GPU RTX (Tensor Core).
torch.backends.cudnn.benchmark = True

TRAIN_DIR = r"D:\Devianest_SkripsiTest\train"
TEST_DIR  = r"D:\Devianest_SkripsiTest\test"

cuda
2.11.0+cu128
True
NVIDIA GeForce RTX 4060


# **Train Augmentation**

In [3]:
# PERBAIKAN: augmentasi dinaikkan dari "light" -> "medium" sesuai rekomendasi sweep
# (flip + rotasi kecil + color jitter ringan). Untuk skin disease, sengaja TIDAK
# pakai augmentasi "strong" (random crop agresif / cutout / blur) karena bisa
# mengubah ciri visual lesi kulit yang justru jadi fitur penting untuk klasifikasi.
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    #transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    #transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(10),
    #transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),  # PERBAIKAN: aktifkan, ringan saja
    transforms.RandomResizedCrop(224, scale=(0.8,1.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    #transforms.RandomErasing(p=0.2, scale=(0.02, 0.1))
])


# **Validation Transform**

In [4]:
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    #transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# **Load Filepaths**

In [5]:
classes = sorted(os.listdir(TRAIN_DIR))
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

num_classes = len(classes)

filepaths = []
labels    = []

for label in classes:
    class_path = os.path.join(TRAIN_DIR, label)
    for img in os.listdir(class_path):
        filepaths.append(os.path.join(class_path, img))
        labels.append(class_to_idx[label])

print("Total Images :", len(filepaths))
print("Classes      :", classes)

Total Images : 15557
Classes      : ['Acne and Rosacea Photos', 'Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions', 'Atopic Dermatitis Photos', 'Bullous Disease Photos', 'Cellulitis Impetigo and other Bacterial Infections', 'Eczema Photos', 'Exanthems and Drug Eruptions', 'Hair Loss Photos Alopecia and other Hair Diseases', 'Herpes HPV and other STDs Photos', 'Light Diseases and Disorders of Pigmentation', 'Lupus and other Connective Tissue diseases', 'Melanoma Skin Cancer Nevi and Moles', 'Nail Fungus and other Nail Disease', 'Poison Ivy Photos and other Contact Dermatitis', 'Psoriasis pictures Lichen Planus and related diseases', 'Scabies Lyme Disease and other Infestations and Bites', 'Seborrheic Keratoses and other Benign Tumors', 'Systemic Disease', 'Tinea Ringworm Candidiasis and other Fungal Infections', 'Urticaria Hives', 'Vascular Tumors', 'Vasculitis Photos', 'Warts Molluscum and other Viral Infections']


# **Dataset Class**

In [6]:
class SkinDataset(Dataset):

    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        image = Image.open(self.filepaths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# **Early Stopping & K-Fold**

In [7]:
class EarlyStopping:
    # PERBAIKAN: kriteria checkpoint/early stopping diganti dari val_loss -> val_f1.
    # Aria (WandB AI) menyarankan checkpoint terbaik dipilih dengan kriteria jelas,
    # misalnya best_val_f1 — supaya model yang disimpan benar-benar yang paling
    # bagus performanya (F1), bukan cuma yang val_loss-nya paling rendah (dua hal
    # ini bisa beda, terutama saat data imbalanced).
    def __init__(self, patience=5):
        self.patience  = patience
        self.best_f1   = -np.inf
        self.counter   = 0

    def step(self, val_f1):
        if val_f1 > self.best_f1:
            self.best_f1 = val_f1
            self.counter = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [8]:

BATCH_SIZE      = 32            # tetap, sesuai rekomendasi Aria (8 atau 16 untuk ViT-Base)
EPOCHS          = 50
EXPERIMENT_NAME = "EXP02_ViT_Base_16_ECA"   # PERBAIKAN: versi ECA dari EXP01, semua hyperparameter identik dengan EXP01 supaya efek ECA terisolasi (tidak ada confound lain)

# ── HYPERPARAMETER KANDIDAT TERBAIK ──────────────────────────────────────────
# PERBAIKAN: kombinasi rekomendasi Claude (analisis overfitting dari EXP01) +
# rekomendasi Aria (WandB AI, dari sweep design). EXP01 full fine-tuning tanpa
# dropout menunjukkan gap besar antara train loss (~0.18) vs val loss (~1.7),
# val loss naik-turun tidak stabil antar fold -> overfitting jelas.
LR               = 3e-5             # FIX (Claude): 1e-4 -> 3e-5. LR 1e-4 terlalu tinggi untuk fine-tune ViT (last_4_blocks + batch kecil), bikin update per-step terlalu agresif -> val loss noisy/oscillating antar epoch.
WEIGHT_DECAY     = 0.01             # PERBAIKAN: 0.01 -> 0.05 (regularisasi lebih kuat, rekomendasi Aria)
DROP_OUT         = 0.1              # PERBAIKAN: 0.0 -> 0.2 (rekomendasi Aria, kandidat utama atasi overfitting)
UNFROZEN_LAYERS  = "last_4_blocks"  # PERBAIKAN: "all" -> "last_4_blocks" (freeze sebagian backbone, rekomendasi Aria)
AUGMENTATION_STRENGTH = "medium"    # PERBAIKAN: augmentasi dinaikkan dari minimal -> medium
LABEL_SMOOTHING = 0.1          # PERBAIKAN: label smoothing ditambahkan (rekomendasi Aria, kandidat utama atasi overfitting)
# PERBAIKAN (tambahan dari Claude, di luar rekomendasi Aria): LR warmup + cosine
# decay. ViT pretrained sensitif di awal training -> warmup linear beberapa
# epoch mencegah update besar yang merusak bobot pretrained, lalu cosine decay
# menurunkan LR bertahap supaya training lebih stabil di akhir.
# WARMUP_EPOCHS    = 5

# ── ECA (Efficient Channel Attention) ────────────────────────────────────────
# Disisipkan di patch embedding ViT (satu-satunya titik insersi yang valid untuk
# modul attention berbasis CNN/channel, karena setelah patch_embed representasi
# sudah berbentuk sequence of tokens, bukan feature map 4D lagi).
USE_ECA          = True
ECA_K_SIZE       = 3    # ukuran kernel conv1d ECA, k=3 default (adaptif terhadap jumlah channel bisa ditambahkan nanti kalau perlu)

run = wandb.init(
    project = "SkinDisease-ViT",
    entity = "devianestnarendra_Team",
    name    = EXPERIMENT_NAME,
    config  = {
        "architecture"   : "ViT-Base/16 (timm: vit_base_patch16_224)",
        "n_folds"        : 5,
        "epochs"         : EPOCHS,
        "batch_size"     : BATCH_SIZE,
        "optimizer"      : "AdamW",
        "lr"             : LR,
        "weight_decay"   : WEIGHT_DECAY,
        "Drop_Out"       : DROP_OUT,
        "unfrozen_layers": UNFROZEN_LAYERS,
        "augmentation_strength": AUGMENTATION_STRENGTH,
        "use_eca"        : USE_ECA,
        "eca_k_size"     : ECA_K_SIZE,
        # "warmup_epochs"  : WARMUP_EPOCHS,            # PERBAIKAN: tambahan, di luar rekomendasi Aria
        "lr_scheduler": "ReduceLROnPlateau",  # PERBAIKAN: tambahan, di luar rekomendasi Aria
        "checkpoint_criteria": "best_val_f1",        # PERBAIKAN: kriteria checkpoint dicatat eksplisit (rekomendasi Aria)
    }
)

print(f"WandB Run : {run.name}")
print(f"URL       : {run.url}")
wandb.run.log_code(".")


WandB Run : EXP02_ViT_Base_16_ECA
URL       : https://wandb.ai/devianestnarendra_Team/SkinDisease-ViT/runs/93t6cfdm


<Artifact source-SkinDisease-ViT-d__Devianest_SkripsiTest_exp02-vit-base-16-eca.ipynb>

# **Training Loop**

In [9]:

# FIX (Claude): AMP (Automatic Mixed Precision) -> sebagian besar operasi jalan di
# float16 (lebih cepat & hemat VRAM di GPU RTX/Tensor Core), sementara update
# gradient tetap presisi (dijaga oleh GradScaler) supaya training tetap stabil.
from torch.cuda.amp import autocast, GradScaler

# ── ECA (Efficient Channel Attention) ────────────────────────────────────────
# Disisipkan sebagai channel-attention setelah proj conv di patch embedding.
# ViT-Base timm: patch_embed.proj adalah Conv2d yang menghasilkan feature map
# (B, C, H, W) sebelum di-flatten jadi token sequence -> ini titik insersi yang
# valid untuk modul attention CNN-native seperti ECA (butuh dimensi channel/
# spatial 4D, tidak bisa dipasang di dalam transformer block yang sudah berupa
# token sequence).
class ECAAttention(nn.Module):
    def __init__(self, channels, k_size=3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (B, C, H, W)
        y = self.avg_pool(x)
        y = y.squeeze(-1).transpose(-1, -2)
        y = self.conv(y)
        y = y.transpose(-1, -2).unsqueeze(-1)
        y = self.sigmoid(y)
        return x * y.expand_as(x)


class ECAPatchEmbed(nn.Module):
    # Bungkus ulang patch_embed asli ViT: proj (Conv2d) -> ECA -> flatten -> norm.
    # Struktur asli timm PatchEmbed: proj -> flatten -> norm. Di sini ECA disisipkan
    # tepat setelah proj (masih 4D), sebelum flatten jadi token sequence.
    def __init__(self, original_patch_embed, eca_k_size=3):
        super().__init__()
        self.proj = original_patch_embed.proj
        self.norm = original_patch_embed.norm
        self.eca = ECAAttention(self.proj.out_channels, k_size=eca_k_size)

    def forward(self, x):
        x = self.proj(x)
        x = self.eca(x)
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x


# PERBAIKAN: fungsi helper untuk strategi freeze/unfreeze layer ViT (rekomendasi
# Aria). timm ViT (vit_base_patch16_224) punya struktur: patch_embed -> blocks
# (ModuleList 12 transformer block) -> norm -> head. "last_N_blocks" berarti
# hanya N block transformer terakhir + norm + head yang ikut dilatih; sisanya
# (patch_embed + block-block awal) dibekukan supaya representasi pretrained
# level rendah tidak rusak saat fine-tuning dataset kecil.
def apply_freeze_strategy(model, strategy: str):
    # default: freeze semua dulu, baru buka sesuai strategi
    for param in model.parameters():
        param.requires_grad = False

    if strategy == "head_only":
        for param in model.head.parameters():
            param.requires_grad = True

    elif strategy == "last_2_blocks":
        for block in model.blocks[-2:]:
            for param in block.parameters():
                param.requires_grad = True
        for param in model.norm.parameters():
            param.requires_grad = True
        for param in model.head.parameters():
            param.requires_grad = True

    elif strategy == "last_4_blocks":
        for block in model.blocks[-4:]:
            for param in block.parameters():
                param.requires_grad = True
        for param in model.norm.parameters():
            param.requires_grad = True
        for param in model.head.parameters():
            param.requires_grad = True

    elif strategy == "all":
        for param in model.parameters():
            param.requires_grad = True

    else:
        raise ValueError(f"Unknown freeze strategy: {strategy}")

    # PERBAIKAN (ECA): submodule ECA di patch_embed selalu dibuka, apapun
    # strategi freeze-nya -> modul baru ini butuh dilatih dari awal (random
    # init), tidak seperti backbone ViT yang sudah pretrained.
    if hasattr(model.patch_embed, "eca"):
        for param in model.patch_embed.eca.parameters():
            param.requires_grad = True

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total     = sum(p.numel() for p in model.parameters())
    print(f"  Freeze strategy   : {strategy}")
    print(f"  Trainable params  : {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.1f}%)")

    return model


# PERBAIKAN (tambahan dari Claude, di luar rekomendasi Aria): LR scheduler
# dengan linear warmup lalu cosine decay. Dipakai per-epoch (bukan per-step)
# supaya cocok dengan struktur training loop yang sudah ada (epoch loop, bukan
# step loop).
# def get_lr_at_epoch(epoch, total_epochs, base_lr, warmup_epochs):
#     import math
#     if epoch < warmup_epochs:
#         # warmup linear: epoch 0 -> lr kecil, naik bertahap ke base_lr
#         return base_lr * (epoch + 1) / warmup_epochs
#     else:
#         # cosine decay setelah warmup selesai
#         progress = (epoch - warmup_epochs) / max(1, (total_epochs - warmup_epochs))
#         return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


fold_results = []
fold_accuracies    = []
fold_precision     = []
fold_recall        = []
fold_f1            = []
all_fold_best_paths = []

all_train_losses = {}
all_val_losses   = {}


for fold, (train_idx, val_idx) in enumerate(skf.split(filepaths, labels)):

    # if fold < 4:
    #     continue

    print(f"\n{'='*50}")
    print(f"  FOLD {fold + 1} / 5")
    print(f"{'='*50}")

    best_val_loss   = np.inf
    best_train_loss = np.inf
    best_val_f1     = -np.inf   # PERBAIKAN: tracking best_val_f1 untuk kriteria checkpoint
    best_model_path = None

 # ── SPLIT ─────────────────────────────────────────────────────────────────
    train_files  = [filepaths[i] for i in train_idx]
    train_labels = [labels[i]    for i in train_idx]
    val_files    = [filepaths[i] for i in val_idx]
    val_labels   = [labels[i]    for i in val_idx]

    # ── DATALOADER ────────────────────────────────────────────────────────────
    train_loader = DataLoader(
        SkinDataset(train_files, train_labels, transform=train_tf),
        batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True
        # FIX (Claude): num_workers 2->4 (percepat data loading, sesuaikan lagi
        # dengan jumlah core CPU kamu kalau masih bottleneck), pin_memory=True
        # mempercepat transfer data CPU->GPU.
    )
    val_loader = DataLoader(
        SkinDataset(val_files, val_labels, transform=eval_tf),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True
    )

    # ── MODEL ─────────────────────────────────────────────────────────────────
    # PERBAIKAN: tambahkan drop_rate=DROP_OUT ke timm.create_model. Ini menambah
    # dropout di classifier head ViT (sebelumnya Drop_Out=0.0 di EXP01, sekarang
    # 0.2 sesuai rekomendasi Aria untuk redam overfitting).
    model = timm.create_model(
        "vit_base_patch16_224",
        pretrained=True,
        num_classes=num_classes,
        drop_rate=DROP_OUT,
        drop_path_rate=0.1
    )

    # PERBAIKAN (ECA): bungkus patch_embed asli dengan ECAPatchEmbed sebelum
    # freeze strategy diterapkan, supaya submodule ECA baru ini kebaca saat
    # apply_freeze_strategy mengecek hasattr(model.patch_embed, "eca").
    if USE_ECA:
        model.patch_embed = ECAPatchEmbed(model.patch_embed, eca_k_size=ECA_K_SIZE)

    # PERBAIKAN: ganti full fine-tuning -> freeze/unfreeze sesuai UNFROZEN_LAYERS
    # (rekomendasi Aria: "last_4_blocks" lebih stabil daripada full fine-tuning
    # untuk dataset yang tidak terlalu besar).
    model = apply_freeze_strategy(model, UNFROZEN_LAYERS)

    model = model.to(device)


    # ── LOSS / OPTIMIZER / SCHEDULER ─────────────────────────────────────────
    class_counts  = np.bincount(train_labels)
    class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
    # FIX (Claude): normalisasi supaya rata-rata weight = 1. Tanpa ini, magnitude
    # weight antar kelas terlalu kecil & timpang -> loss "melompat" tergantung
    # komposisi kelas tiap batch, jadi salah satu penyebab val loss noisy.
    class_weights = class_weights / class_weights.sum() * num_classes

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device),
        label_smoothing=LABEL_SMOOTHING
    )

    # PERBAIKAN: lr=3e-5 -> LR (2e-5), weight_decay=1e-2 -> WEIGHT_DECAY (0.05).
    # filter(requires_grad) tetap dipakai -> otomatis hanya optimize parameter
    # yang dibuka oleh apply_freeze_strategy().
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',        # karena monitor F1
    factor=0.1,        # LR dikali 0.5 jika stagnan
    patience=2,        # tunggu 2 epoch
    threshold=1e-4,
    min_lr=1e-7
)
    
    # FIX (Claude): GradScaler untuk AMP -> scale loss sebelum backward supaya
    # gradient kecil di float16 tidak underflow jadi nol.
    scaler = GradScaler()

    early_stopping = EarlyStopping(patience=5)
    best_model_wts = copy.deepcopy(model.state_dict())

    train_losses = []
    val_losses   = []

    # ── EPOCH LOOP ────────────────────────────────────────────────────────────
    for epoch in range(EPOCHS):

        # PERBAIKAN: set LR sesuai schedule warmup + cosine decay sebelum epoch
        # berjalan. current_lr dihitung per-epoch lalu diterapkan ke optimizer.

        
        # current_lr = get_lr_at_epoch(epoch, EPOCHS, LR, WARMUP_EPOCHS)
        # for param_group in optimizer.param_groups:
        #     param_group['lr'] = current_lr

        print(f"\nEpoch {epoch + 1}/{EPOCHS} (LR: {optimizer.param_groups[0]['lr']:.2e})")

        # TRAIN
        model.train()
        train_loss = 0
        for images, targets in tqdm(train_loader, desc="Train"):
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad()
            # FIX (Claude): forward pass di dalam autocast -> otomatis pilih
            # float16/float32 per operasi. backward & step lewat scaler biar
            # gradient tetap akurat walau sebagian forward pakai float16.
            with autocast():
                loss = criterion(model(images), targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()

        # VALIDATION
        model.eval()
        val_loss = 0
        preds, trues = [], []
        with torch.no_grad():
            for images, targets in tqdm(val_loader, desc="Val"):
                images, targets = images.to(device), targets.to(device)
                # FIX (Claude): autocast juga di validation -> ikut lebih cepat,
                # aman karena tidak ada backward/gradient di sini.
                with autocast():
                    outputs = model(images)
                    v_loss  = criterion(outputs, targets)
                val_loss += v_loss.item()
                preds.extend(outputs.argmax(1).cpu().numpy())
                trues.extend(targets.cpu().numpy())

        # METRICS
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss   = val_loss   / len(val_loader)

        acc       = accuracy_score(trues, preds)
        precision = precision_score(trues, preds, average='weighted', zero_division=0)
        recall    = recall_score(trues, preds, average='weighted', zero_division=0)
        f1        = f1_score(trues, preds, average='weighted', zero_division=0)
        scheduler.step(f1)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        print(f"Train Loss : {avg_train_loss:.4f} | Val Loss  : {avg_val_loss:.4f}")
        print(f"Accuracy   : {acc:.4f}  | Precision : {precision:.4f}")
        print(f"Recall     : {recall:.4f}  | F1 Score  : {f1:.4f}")

        # ── WANDB LOG PER EPOCH ───────────────────────────────────────────────
        # Panel Loss      → fold_N/train_loss, fold_N/val_loss
        # Panel Accuracy  → fold_N/accuracy
        # Panel Precision → fold_N/precision
        # Panel Recall    → fold_N/recall
        # Panel F1 Score  → fold_N/f1_score
        # Panel LR        → fold_N/lr
        # Semua pakai key "epoch" sebagai x-axis bersama
        wandb.log({
            "epoch"                    : epoch + 1,

            f"fold_{fold+1}/train_loss": avg_train_loss,
            f"fold_{fold+1}/val_loss"  : avg_val_loss,

            f"fold_{fold+1}/accuracy"  : acc,
            f"fold_{fold+1}/precision" : precision,
            f"fold_{fold+1}/recall"    : recall,
            f"fold_{fold+1}/f1_score"  : f1,

            f"fold_{fold+1}/lr"        : optimizer.param_groups[0]['lr'],

        })



        # SAVE BEST MODEL
        # PERBAIKAN: kriteria checkpoint diganti dari "val_loss terendah" menjadi
        # "val_f1 tertinggi" (rekomendasi Aria: checkpoint_criteria = best_val_f1).
        # val_loss & train_loss tetap dicatat untuk laporan, tapi bukan lagi
        # acuan penyimpanan model terbaik.
        if f1 > best_val_f1:
            best_val_f1     = f1
            best_val_loss   = avg_val_loss
            best_train_loss = avg_train_loss

            # FIX (Claude): path /kaggle/working tidak ada di lokal (Windows) -> ganti
            # ke folder lokal relatif, dibuat otomatis kalau belum ada.
            os.makedirs("outputs", exist_ok=True)
            save_path      = f"outputs/model_fold_{fold + 1}.pth"
            torch.save({
                "model_state_dict": model.state_dict(),
                "val_loss"        : avg_val_loss,
                "f1"              : f1,
                "fold"            : fold + 1
            }, save_path)
            best_model_path = save_path
            best_model_wts  = copy.deepcopy(model.state_dict())
            print(f"  ✓ Model saved (best val_f1: {best_val_f1:.4f}) → {save_path}")

        # PERBAIKAN: EarlyStopping.step() sekarang menerima val_f1, bukan val_loss
        # (selaras dengan kriteria checkpoint di atas).
        if early_stopping.step(f1):
            print("Early Stopping Triggered")
            break

    # ── SIMPAN HISTORY ────────────────────────────────────────────────────────
    all_train_losses[fold + 1] = train_losses
    all_val_losses[fold + 1]   = val_losses

    # ── PLOT LOSS CURVE PER FOLD ──────────────────────────────────────────────
    epochs_ran = range(1, len(train_losses) + 1)
    fig, ax    = plt.subplots(figsize=(8, 5))
    ax.plot(epochs_ran, train_losses, label='Train Loss', marker='o', markersize=3)
    ax.plot(epochs_ran, val_losses,   label='Val Loss',   marker='o', markersize=3)
    ax.set_title(f'Fold {fold + 1} — Loss Curve')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

    os.makedirs("outputs", exist_ok=True)
    curve_path = f"outputs/Fold_{fold + 1}_Loss_Curve.png"
    fig.savefig(curve_path, dpi=150, bbox_inches='tight')
    wandb.log({f"Loss_Curve/Fold_{fold+1}": wandb.Image(curve_path)})
    plt.close(fig)
    print(f"  ✓ Loss curve saved → {curve_path}")

    # ── UPLOAD MODEL ARTIFACT ─────────────────────────────────────────────────
    if best_model_path:
        artifact = wandb.Artifact(name=f"model-fold-{fold+1}", type="model")
        artifact.add_file(best_model_path)
        wandb.log_artifact(artifact)
        all_fold_best_paths.append(best_model_path)

    # ── FINAL EVALUATION FOLD (pakai best model) ──────────────────────────────
    if best_model_path:
        model.load_state_dict(
            torch.load(best_model_path, map_location=device)["model_state_dict"]
        )

    model.eval()
    final_preds, final_trues = [], []
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)
            final_preds.extend(model(images).argmax(1).cpu().numpy())
            final_trues.extend(targets.cpu().numpy())

    print("\nClassification Report")
    print(classification_report(final_trues, final_preds, target_names=classes, zero_division=0))

    fold_acc  = accuracy_score(final_trues, final_preds)
    fold_prec = precision_score(final_trues, final_preds, average='weighted', zero_division=0)
    fold_rec  = recall_score(final_trues, final_preds, average='weighted', zero_division=0)
    fold_f1_  = f1_score(final_trues, final_preds, average='weighted', zero_division=0)

    fold_accuracies.append(fold_acc)
    fold_precision.append(fold_prec)
    fold_recall.append(fold_rec)
    fold_f1.append(fold_f1_)

    fold_results.append({
        "Fold": fold + 1,
        "Train_Loss": best_train_loss,
        "Val_Loss": best_val_loss,
        "Accuracy": fold_acc,
        "Precision": fold_prec,
        "Recall": fold_rec,
        "F1": fold_f1_
})



    # ==========================================
    # CONFUSION MATRIX PER FOLD
    # ==========================================
    cm = confusion_matrix(final_trues, final_preds)

    fig, ax = plt.subplots(figsize=(12, 12))

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=classes
    ).plot(
        ax=ax,
        cmap="Blues",
        xticks_rotation=90
    )

    plt.tight_layout()

    os.makedirs("outputs", exist_ok=True)

    cm_path = f"outputs/Fold_{fold+1}_ConfusionMatrix.png"
    plt.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    
    # ── WANDB LOG FINAL METRICS FOLD ─────────────────────────────────────────
    # Panel Final Accuracy  → fold_N/final_accuracy
    # Panel Final Precision → fold_N/final_precision
    # Panel Final Recall    → fold_N/final_recall
    # Panel Final F1        → fold_N/final_f1
    wandb.log({
        f"fold_{fold+1}/final_accuracy": fold_acc,
        f"fold_{fold+1}/final_precision": fold_prec,
        f"fold_{fold+1}/final_recall": fold_rec,
        f"fold_{fold+1}/final_f1": fold_f1_,
        f"fold_{fold+1}/confusion_matrix": wandb.Image(cm_path)
    })

    print(f"\nFold {fold+1} selesai — Acc: {fold_acc:.4f} | F1: {fold_f1_:.4f}")

    # bersihkan GPU memory antar fold
    del model, optimizer, best_model_wts
    torch.cuda.empty_cache()


results_df = pd.DataFrame(fold_results)

results_df.loc[len(results_df)] = {
    "Fold": "Mean",
    "Train_Loss": results_df["Train_Loss"].mean(),
    "Val_Loss": results_df["Val_Loss"].mean(),
    "Accuracy": np.mean(fold_accuracies),
    "Precision": np.mean(fold_precision),
    "Recall": np.mean(fold_recall),
    "F1": np.mean(fold_f1)
}

csv_path = "outputs/KFold_Summary.csv"
results_df.to_csv(csv_path, index=False)

artifact = wandb.Artifact(
    "kfold-summary",
    type="results"
)

artifact.add_file(csv_path)

wandb.log_artifact(artifact)



  FOLD 1 / 5


  Freeze strategy   : last_4_blocks
  Trainable params  : 28,370,714 / 85,816,346 (33.1%)


C:\Users\UNIDA\AppData\Local\Temp\ipykernel_11396\3208401025.py:221: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()



Epoch 1/50 (LR: 3.00e-05)


Train:   0%|          | 0/389 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_11396\3208401025.py:251: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/98 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_11396\3208401025.py:267: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val: 100%|██████████| 98/98 [00:23<00:00,  4.10it/s]


...

Train:   0%|          | 0/389 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_11396\3208401025.py:251: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/98 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_11396\3208401025.py:267: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val: 100%|██████████| 98/98 [00:26<00:00,  3.72it/s]


Train Loss : 1.1009 | Val Loss  : 2.1378
Accuracy   : 0.6185  | Precision : 0.6232
Recall     : 0.6185  | F1 Score  : 0.6184
  ✓ Model saved (best val_f1: 0.6184) → outputs/model_fold_5.pth

Epoch 50/50 (LR: 3.00e-06)


Train:   0%|          | 0/389 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_11396\3208401025.py:251: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/98 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_11396\3208401025.py:267: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val: 100%|██████████| 98/98 [00:26<00:00,  3.68it/s]


Train Loss : 1.0924 | Val Loss  : 2.1419
Accuracy   : 0.6146  | Precision : 0.6196
Recall     : 0.6146  | F1 Score  : 0.6148
  ✓ Loss curve saved → outputs/Fold_5_Loss_Curve.png

Classification Report
                                                                    precision    recall  f1-score   support

                                           Acne and Rosacea Photos       0.76      0.82      0.79       168
Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions       0.67      0.64      0.65       229
                                          Atopic Dermatitis Photos       0.68      0.61      0.65        98
                                            Bullous Disease Photos       0.51      0.56      0.53        90
                Cellulitis Impetigo and other Bacterial Infections       0.48      0.42      0.45        57
                                                     Eczema Photos       0.64      0.65      0.65       247
                                      Exan

<Artifact kfold-summary>

# **Grafik Gabungan & Final Summary**

In [10]:
# ── GRAFIK GABUNGAN SEMUA FOLD ────────────────────────────────────────────────
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for i, fold_n in enumerate(all_train_losses.keys()):
    ep = range(1, len(all_train_losses[fold_n]) + 1)
    c  = colors[(fold_n - 1) % len(colors)]
    axes[0].plot(ep, all_train_losses[fold_n], label=f'Fold {fold_n}', color=c, marker='o', markersize=3)
    axes[1].plot(ep, all_val_losses[fold_n],   label=f'Fold {fold_n}', color=c, marker='o', markersize=3)

for ax, title in zip(axes, ['Train Loss — Semua Fold', 'Val Loss — Semua Fold']):
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Perbandingan Loss Semua Fold', fontsize=14, fontweight='bold')
plt.tight_layout()

os.makedirs("outputs", exist_ok=True)
combined_path = "outputs/All_Folds_Loss_Curve.png"
fig.savefig(combined_path, dpi=150, bbox_inches='tight')
wandb.log({"Loss_Curve/All_Folds_Combined": wandb.Image(combined_path)})
plt.close(fig)
print(f"✓ Grafik gabungan disimpan → {combined_path}")

# ── SUMMARY METRICS ───────────────────────────────────────────────────────────
print("\n" + "="*50)
print("  FINAL RESULT — ALL FOLDS")
print("="*50)
print(f"Mean Accuracy  : {np.mean(fold_accuracies):.4f} ± {np.std(fold_accuracies):.4f}")
print(f"Mean Precision : {np.mean(fold_precision):.4f} ± {np.std(fold_precision):.4f}")
print(f"Mean Recall    : {np.mean(fold_recall):.4f} ± {np.std(fold_recall):.4f}")
print(f"Mean F1 Score  : {np.mean(fold_f1):.4f} ± {np.std(fold_f1):.4f}")

# ── WANDB LOG SUMMARY ─────────────────────────────────────────────────────────
# Panel Summary → summary/mean_accuracy, summary/mean_precision, dst.
wandb.log({
    "summary/mean_accuracy"  : np.mean(fold_accuracies),
    "summary/mean_precision" : np.mean(fold_precision),
    "summary/mean_recall"    : np.mean(fold_recall),
    "summary/mean_f1"        : np.mean(fold_f1),
    "summary/std_accuracy"   : np.std(fold_accuracies),
    "summary/std_f1"         : np.std(fold_f1),
})



✓ Grafik gabungan disimpan → outputs/All_Folds_Loss_Curve.png

  FINAL RESULT — ALL FOLDS
Mean Accuracy  : 0.5992 ± 0.0116
Mean Precision : 0.6064 ± 0.0093
Mean Recall    : 0.5992 ± 0.0116
Mean F1 Score  : 0.5997 ± 0.0110


# **Test Evaluation**

In [11]:
best_overall_path = all_fold_best_paths[fold_f1.index(max(fold_f1))]
print(f"Best model path : {best_overall_path}")
print(f"Best F1         : {max(fold_f1):.4f}")

test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_tf)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# PERBAIKAN: di versi ResNet50, variabel `model` yang dipakai di sini adalah sisa
# `model` dari iterasi fold terakhir Cell 17 (kebetulan arsitekturnya sama tiap
# fold, jadi tidak error, tapi rapuh). Di sini dibuat eksplisit: instance model
# ViT-Base baru, lalu load bobot terbaik -> lebih jelas & tidak tergantung state
# sisa loop sebelumnya.
model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=num_classes
)

# PERBAIKAN (ECA): rekonstruksi patch_embed dengan ECA SEBELUM load_state_dict,
# karena checkpoint menyimpan bobot ECAPatchEmbed (proj + eca + norm), bukan
# PatchEmbed bawaan timm -> kalau tidak dibungkus dulu, load_state_dict akan
# error key mismatch (missing "patch_embed.eca.*").
if USE_ECA:
    model.patch_embed = ECAPatchEmbed(model.patch_embed, eca_k_size=ECA_K_SIZE)

model = model.to(device)

checkpoint = torch.load(best_overall_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


Best model path : outputs/model_fold_5.pth
Best F1         : 0.6180


VisionTransformer(
  (patch_embed): ECAPatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
    (eca): ECAAttention(
      (avg_pool): AdaptiveAvgPool2d(output_size=1)
      (conv): Conv1d(1, 1, kernel_size=(3,), stride=(1,), padding=(1,), bias=False)
      (sigmoid): Sigmoid()
    )
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, e

# **Test Confusion Matrix**

In [12]:
y_true, y_pred = [], []

with torch.no_grad():
    for images, lbs in tqdm(test_loader, desc="Test"):
        images  = images.to(device)
        outputs = model(images)
        y_true.extend(lbs.cpu().numpy())
        y_pred.extend(outputs.argmax(1).cpu().numpy())

acc       = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
recall    = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1        = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=classes, zero_division=0))

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(15, 15))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=classes
).plot(
    ax=ax,
    cmap="Blues",
    xticks_rotation=90
)

plt.tight_layout()

os.makedirs("outputs", exist_ok=True)

cm_path = "outputs/Test_Confusion_Matrix.png"
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.close(fig)

wandb.log({
    "Test/Confusion_Matrix": wandb.Image(cm_path),
    "test/accuracy": acc,
    "test/precision": precision,
    "test/recall": recall,
    "test/f1": f1
})

# ==========================================
# TEST RESULT CSV
# ==========================================

test_results_df = pd.DataFrame({
    "Filename": [test_dataset.samples[i][0] for i in range(len(y_true))],
    "True_Label": [classes[i] for i in y_true],
    "Predicted_Label": [classes[i] for i in y_pred],
    "Correct": np.array(y_true) == np.array(y_pred)
})


test_summary_df = pd.DataFrame([{
    "Accuracy": acc,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
}])

os.makedirs("outputs", exist_ok=True)

summary_path = "outputs/Test_Summary.csv"
test_summary_df.to_csv(summary_path, index=False)

csv_test_path = "outputs/Test_Result.csv"
test_results_df.to_csv(csv_test_path, index=False)

print(f"Test CSV saved -> {csv_test_path}")


# ==========================================
# UPLOAD TEST CSV KE WANDB
# ==========================================

artifact = wandb.Artifact(
    name="test-results",
    type="results"
)

artifact.add_file(csv_test_path)
artifact.add_file(summary_path)

wandb.log_artifact(artifact)

wandb.finish()
print("\nWandB run selesai.")

Test: 100%|██████████| 126/126 [00:44<00:00,  2.85it/s]


Accuracy  : 0.6319
Precision : 0.6369
Recall    : 0.6319
F1-Score  : 0.6313

                                                                    precision    recall  f1-score   support

                                           Acne and Rosacea Photos       0.83      0.88      0.85       312
Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions       0.66      0.65      0.65       288
                                          Atopic Dermatitis Photos       0.59      0.60      0.60       123
                                            Bullous Disease Photos       0.57      0.56      0.57       113
                Cellulitis Impetigo and other Bacterial Infections       0.38      0.38      0.38        73
                                                     Eczema Photos       0.68      0.65      0.66       309
                                      Exanthems and Drug Eruptions       0.47      0.55      0.51       101
                 Hair Loss Photos Alopecia and other Hair 

epoch,▁▂▄▅▅▆▆▁▁▂▄▄▅▅▆███▁▄▅▅▆▆▇▃▃▄▅▅█▁▃▃▃▄▄▄▄█
fold_1/accuracy,▁▃▃▄▄▄▅▅▆▅▆▆▆▇▇▇▇▇▇▇▇▇█▇▇▇██████████████
fold_1/f1_score,▁▂▃▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇█▇████████████████
fold_1/final_accuracy,▁
fold_1/final_f1,▁
fold_1/final_precision,▁
fold_1/final_recall,▁
fold_1/lr,█████████████████████████▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁
fold_1/precision,▁▃▄▄▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇█▇████████████████
fold_1/recall,▁▃▃▄▄▄▅▅▆▅▆▆▆▇▇▇▇▇▇▇▇▇█▇▇▇██████████████
+56,...



WandB run selesai.
